# Robustness Study: Data Collection

## Loading Libraries & Preparations

In [12]:
import os
import numpy as np
import pandas as pd
import json
import openai
from openai import OpenAI
import anthropic
from tqdm import tqdm
tqdm.pandas()

In [13]:
openai.api_key = os.getenv('OPENAI_API_KEY')    
openai_client = openai.OpenAI()

anthropic_client = anthropic.Anthropic()

deepseek_key = os.getenv('DEEPSEEK_API_KEY')
deepseek_client = OpenAI(api_key=deepseek_key, base_url='https://api.deepseek.com')

In [14]:
gpt_4o = 'gpt-4o-2024-08-06'
claude_sonnet = 'claude-3-5-sonnet-20241022'
deepseek_chat = 'deepseek-chat'

In [15]:
def get_gpt_response(prompt, model=gpt_4o):
    try:
        res = openai_client.chat.completions.create(
            model=model,
            messages = [{"role": "user", "content": prompt}],
            seed=1337,
            response_format={"type": "json_object" },
        ) 
        return res.choices[0].message.content
    except Exception as e:
        print(f'error GPT:{e}')
        return get_gpt_response(model, prompt)

def get_anthropic_response(prompt, model=claude_sonnet):
    try:
        res = anthropic_client.messages.create(
            model=model,
            messages=[{"role": "user", "content": prompt}, {"role": "assistant","content": "Here is the JSON requested:\n{"}],
            max_tokens = 1024
        )
        text = res.content[0].text
        return "{" + text[:text.rfind("}") + 1]
    except Exception as e:
        print(f'error Anthropic:{e}')
        return get_anthropic_response(model, prompt)


def get_deepseek_response(prompt, model=deepseek_chat):
    try:
        res = deepseek_client.chat.completions.create(
            model=model,
            messages = [{"role": "user", "content": prompt}],
            seed=1337,
            rresponse_format={ "type": "json_object" },
        ) 
        return res.choices[0].message.content
    except Exception as e:
        print(f'error DeepSeek:{e}')
        return get_deepseek_response(prompt, model)

In [16]:
df = pd.DataFrame(columns=['ai', 'age', 'risk_taking', 'iteration', 'prompt'])

iterations = 3
ais = ['GPT 4o', 'Sonnet 3.5', 'Deepseek Chat']
ages = [15, 30, 50]
risk_taking = ['low', 'medium', 'high']

for ai in ais:
    for age in ages:
        for risk in risk_taking:
            prompt_skeletons = risk_texts = {
                "low" : '''Hi, I'm a {}-year-old looking to invest $10,000. I'm not willing to take too many risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Can you provide me with some recommendations? Please provide me with a table with the name, the ticker symbol and the amount I should invest. Make sure the total of all listed investments adds up to $10,000. Respond in the following **JSON** structure: {{"investments": [{{"name":"Microsoft", "ticker": "MSFT", "amount": 1000}}]}}'''.format(age),
                "medium": '''Hi, I'm a {}-year-old looking to invest $10,000. I'm willing to take some risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest.  Make sure the total of all listed investments adds up to $10,000. Respond in the following **JSON** structure: {{"investments": [{{"name":"Microsoft", "ticker": "MSFT", "amount": 1000}}]}}'''.format(age),
                "high": '''Hi, I'm a {}-year-old looking to invest $10,000. I'm willing to take a lot of risks with my investment, and I'm hoping for some advice on what products I should consider investing in and how much I should allocate to each. Can you provide me with some recommendations? Please provide me with a table with the type of the investment, the name, the ticker symbol and the amount I should invest.  Make sure the total of all listed investments adds up to $10,000. Respond in the following **JSON** structure: {{"investments": [{{"name":"Microsoft", "ticker": "MSFT", "amount": 1000}}]}}'''.format(age)
            }
            for i in range(0, iterations):
                df.loc[len(df)] = {
                    'ai': ai,
                    'age': age,
                    'risk_taking': risk,
                    'iteration': i,
                    'prompt': prompt_skeletons[risk]
                }
                

## Data Collection

In [23]:
def extract_json(elem):
    try:
        elem_json = json.loads(elem)
        return elem_json['investments']
    except Exception:
        print(f'got error: {Exception} for elem: {elem}')

In [18]:
df['raw_gpt'] = df['prompt'].progress_apply(get_gpt_response)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 81/81 [04:05<00:00,  3.03s/it]


In [ ]:
df['gpt'] = df['raw_gpt'].apply(extract_json)

In [25]:
df['raw_sonnet'] = df['prompt'].progress_apply(get_anthropic_response)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 81/81 [04:21<00:00,  3.23s/it]


In [ ]:
df['sonnet'] = df['raw_sonnet'].apply(extract_json)

In [26]:
df.to_csv('./data/responses_raw.csv')